# 11 — Memory transformations: the L1 optimizers

Notebook 10 ended on a measurement and a cliffhanger: the L1 footprint
simulator, run on GPT-2, shows the forward+backward *joint* peaking far above
the forward pass — re-measured below on the region joint, a forward peak of
**6992 B** against a joint peak of **18624 B**, **2.66×** higher, because the
backward pass keeps forward activations alive to consume them. That ratio *is*
the reason L1 has more than a measuring stick. This notebook is the three
transforms that push the number back down, each one the same loop:
**measure → transform → re-measure**, with the reference layer's `peak_memory`
(bytes) and `ops_count` (FLOPs) as the two dials on the trade.

1. **Requested-gradients DCE** — *don't save what no gradient needs.* "Which
   gradients do I care about" turns out to be plain backward reachability on
   the joint DAG. No special cases, no "frozen layer" mode: freezing a
   parameter is dropping its gradient from the keep-set, and the
   saved-activation set shrinks by itself.
2. **Min-cut activation checkpointing** — *recompute instead of store.* The
   saved-for-backward set is a minimum cut between the primal inputs and the
   backward's consumers, with edge capacities in *exact bytes*. Three facts our
   representation knows — closed forms are free, views are uncuttable,
   contractions are recompute-banned — sharpen the classic AOTAutograd
   partitioner into something that reads its cost model straight off the IR.
3. **Segmented (checkpointed) fold adjoints** — *the same idea, inside a
   loop.* A `fold`'s backward can keep only segment-boundary states and
   recompute each segment's trajectory just-in-time. Sweeping the segment count
   K over a T=12 time-stepper reproduces Chen's √T heuristic from our own
   primitives — a memory/recompute curve with a measured minimum.

Everything here is a region-to-region rewrite: the input is a dialect Region,
the output is a dialect Region, and `grad`'s returned gradient names survive
untouched, so the measurements are apples-to-apples. One honesty note before
the numbers: a region *is* its yield-reachable graph, so the joint that `grad`
returns is **implicitly DCE'd** — adjoint work that no yielded gradient needs
never exists in the first place — and the byte figures below are the region
joint's own, printed as measured.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from pdum.dsl.ir import Builder, Region
from pdum.dsl.ops import CORE_OPS
from pdum.dsl.types import Literal
from pdum.tl import Tensor, pointwise, red, reduce
from pdum.tl.autodiff import grad
from pdum.tl.dialect import TL_OPS, region_names, run_named, tensor_type_of_layout, walk_region
from pdum.tl.lifting import lift_step
from pdum.tl.markers import exp
from pdum.tl.memory import peak_memory
from pdum.tl.opcount import ops_count
from pdum.tl.transforms import checkpoint, dce
from pdum.tl.zoo import gpt2

In [2]:
def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def nvalues(region):
    # named values only: params + tl ops. Yields, deferred consts, and the
    # yielded tuple are plumbing under the naming law — never counted.
    yielded = region.body[-1].args[0]
    return sum(
        1
        for nd in walk_region(region)
        if not (nd.op in ("core.yield", "core.const") or (nd.op == "core.tuple" and nd is yielded))
    )


def loss_head(m):
    # Append a scalar training loss (sum of squared outputs) so `grad` seeds
    # itself with 1 — the ordinary "one number to differentiate" endpoint.
    # On the region face this is an append-and-re-yield: re-open the model's
    # region with a Builder, square the output node, sum-reduce it, and yield
    # the new scalar; the model's names map extends over the two new nodes.
    # The forward peak is unchanged (the extra ops are a pointwise + a reduce).
    out = next(n for n in walk_region(m.region) if m.names.get(id(n)) == m.out)
    dims = tuple(d.name for d in out.type.layout.dims)
    b = Builder({**CORE_OPS, **TL_OPS})
    zsq = b.emit("tl.pointwise", out, out, f="mul")
    zloss = b.emit("tl.reduce", zsq, f="sum", dims=dims)
    region = Region(params=m.region.params, body=(b.emit("core.yield", zloss),))
    return region, {**m.names, id(zsq): "zsq", id(zloss): "zloss"}

## Move 1 — Requested-gradients DCE

Build the GPT-2 joint the ordinary way: a scalar training loss, then `grad`.
`grad` returns a `RegionGrad` whose joint region yields the loss *and* a
gradient for every differentiable input — the tied embedding `wte`, the
learned positions `wpe`, and all the block weights (the integer token `ids`
are gradient-free). That is why it is big.

In [3]:
m = gpt2()
lregion, lnames = loss_head(m)
rg = grad(lregion, "zloss", m.inputs, names=lnames)  # a RegionGrad: joint + names + grads

fwd_peak = peak_memory(lregion, m.inputs, names=lnames).peak_bytes
joint_peak = peak_memory(rg.region, m.inputs, names=rg.names).peak_bytes
print(f"forward loss     : {nvalues(lregion):>4} values   peak {fwd_peak:>6} B")
print(f"forward+backward : {nvalues(rg.region):>4} values   peak {joint_peak:>6} B   "
      f"({joint_peak / fwd_peak:.2f}x)")
ngrads = sum(g is not None for g in rg.grads.values())
print(f"the joint computes gradients for {ngrads} of {len(m.inputs)} inputs "
      f"(every weight; the integer ids are gradient-free)")

forward loss     :  208 values   peak   6992 B
forward+backward :  807 values   peak  18624 B   (2.66x)
the joint computes gradients for 28 of 29 inputs (every weight; the integer ids are gradient-free)


Now the DCE thesis in one line. Suppose the only gradient you actually want is
`dL/dwte` — you are, say, probing the embedding, or every other weight is
frozen. `dce(region, keep, names=...)` re-yields exactly the kept names and
lets yield-reachability do the pruning: any value that does not transitively
feed something kept drops out. The other weight-gradient contractions
(`activationᵀ · upstream`, the expensive part) never reach `dL/dwte`, so they
vanish — and with them the activations they were the only consumers of.

In [4]:
keep_wte = dce(rg.region, (rg.grads["wte"], "zloss"), names=rg.names)
peak_wte = peak_memory(keep_wte, m.inputs, names=rg.names).peak_bytes
print(f"request only dL/dwte : {nvalues(keep_wte):>4} values   peak {peak_wte:>6} B")
print(f"  dropped {nvalues(rg.region) - nvalues(keep_wte)} backward values; "
      f"{joint_peak - peak_wte} B off the peak")

full = run_named(rg.region, m.inputs, rg.names)
only = run_named(keep_wte, m.inputs, rg.names)  # the same names map serves the pruned region
same = np.allclose(
    only[rg.grads["wte"]].to_numpy(order=("v", "d")),
    full[rg.grads["wte"]].to_numpy(order=("v", "d")),
    rtol=1e-12,
)
print("  dL/dwte is bit-for-bit identical to the full joint:", same)

request only dL/dwte :  694 values   peak  18144 B
  dropped 113 backward values; 480 B off the peak


  dL/dwte is bit-for-bit identical to the full joint: True


**Freezing is not a mode — it is a keep-set.** GPT-2 here is two blocks
(`h.0`, `h.1`) plus the tied embedding and the layer-norm head. "Freeze block
`h.0`" means: request every gradient *except* the `h.0.*` weight gradients. It
is the same reachability query with a different keep-set. The extremes are
"train everything" (all 28 gradients) and "only `dL/dwte`" (freeze everything
else) — no special case sits between them. The *work* falls monotonically as
you request less; the default-schedule *peak* mostly follows, but not
monotonically — the peak is the **schedule's** number (notebook 10's thesis),
and a pruned DAG re-schedules, so freezing one block can land a touch higher
even as the value count falls.

In [5]:
def keepset(freeze=()):
    kept = tuple(g for n in m.inputs
                 if not any(n.startswith(p) for p in freeze)
                 and (g := rg.grads[n]) is not None)  # ids: integer input, gradient-free
    return dce(rg.region, kept + ("zloss",), names=rg.names)


rows = [
    ("train everything         ", rg.region),
    ("freeze block h.0         ", keepset(freeze=("h.0.",))),
    ("only dL/dwte (freeze all)", keep_wte),
]
for label, region in rows:
    pk = peak_memory(region, m.inputs, names=rg.names).peak_bytes
    print(f"{label}: {nvalues(region):>4} values   peak {pk:>6} B")

train everything         :  807 values   peak  18624 B
freeze block h.0         :  755 values   peak  18784 B
only dL/dwte (freeze all):  694 values   peak  18144 B


## Move 2 — Min-cut activation checkpointing

DCE removes work you never wanted. Checkpointing trades work you *do* want —
storage for recomputation. The joint splits at the loss into forward and
backward; the backward reads a set **R** of forward values, and naively all of
R stays live across the boundary. Instead we pick a **saved** set S and
recompute everything else in R from `S ∪ inputs` — on the region face the
recomputed values are fresh *clones*, visited by the walk order just before
their first consumer, so they do not all coexist.

Choosing S is the **min-cut formulation** (REPRESENTATIONS.md's prior-art map:
this is AOTAutograd's partitioner) — a minimum cut between *sources* (primal
inputs + recompute-banned outputs) and a *sink* fed by R, with node capacities
= exact bytes from the layout shadows. Start with the smallest interesting
chain: square, exponentiate, contract, square-the-scalar.

In [6]:
def chain():
    # x -> sq (mul) -> e (exp) -> r (reduce, a contraction) -> loss = r*r
    def body(x):
        sq = x * x                    # 64 B
        e = pointwise(exp, sq)        # 64 B
        r = reduce(red.sum, e, "i")   # 8 B — the contraction
        loss = r * r                  # scalar
        return loss

    inputs = {"x": T(0.1 * np.arange(8.0), ("i",))}
    return lift_step(body, x=inputs["x"].layout), inputs


cls, cin = chain()
crg = grad(cls.region, "loss", cin, names=cls.names)
ck = checkpoint(crg.region, "loss", names=crg.names)
print("saved      :", list(ck.saved))
print("recomputed :", ck.recomputed)
print(f"boundary   : {ck.bytes_before} B naively  ->  {ck.bytes_after} B saved")

saved      : [('r', 8)]
recomputed : ('sq',)
boundary   : 72 B naively  ->  8 B saved


The min cut saves exactly `r` — 8 bytes. Naively the boundary would hold the
big pointwise `sq` (64 B) *and* `r` (72 B total); instead `sq` is recomputed
from the free-to-keep input `x`, and only the banned reduce output `r` is
stored. (`e` never enters the boundary at all: the backward re-applies `exp`
to `sq` — the adjoint reads `sq`, not `e` — and reduce-sum's adjoint is a
plain broadcast of the cotangent.) Recomputation on the region face is node
**cloning** under fresh `.rc` names: `sq.rc` and `sq` are **two node objects,
one semantic value** — the `name ≠ value` price REPRESENTATIONS.md predicted,
harmless for run/measure.

In [7]:
print("the rewritten backward recomputes the cheap chain as clones under fresh .rc names:")
print("    recomputed  :", ck.recomputed)
print("    clone names :", sorted(nm for nm in ck.names.values() if ".rc" in nm))

gj = run_named(crg.region, cin, crg.names)[crg.grads["x"]].to_numpy()
gc = run_named(ck.region, cin, ck.names)[crg.grads["x"]].to_numpy()
print("\ndL/dx identical (sq and sq.rc are one value under two names):",
      np.allclose(gj, gc, rtol=1e-12))

the rewritten backward recomputes the cheap chain as clones under fresh .rc names:
    recomputed  : ('sq',)
    clone names : ['sq.rc']

dL/dx identical (sq and sq.rc are one value under two names): True


**The three representation sharpenings.** The classic partitioner treats every
tensor as an opaque blob with one size. Ours reads three structural facts off
the IR, which is what makes the cut land where it does:

- **Closed forms cost 0 — free to "save".** `iota`, `const`, and
  masks-held-as-guards are computed from position, not stored, so their cut
  capacity is zero. They never enter the budget.
- **Views are uncuttable.** A slice/shift/repeat is an alias with no bytes of
  its own; saving it would pin its root. So a view gets **∞** capacity at the
  view — the cut must fall on its *root* (or recompute the view, which is
  free).
- **Contractions are recompute-banned = fresh demand sources.** `reduce`,
  `scan`, `fold` are the operations whose recompute would double real FLOPs, so
  by default they *source* demand into the cut: the saved set must land
  at-or-after them. Pointwise chains and layout ops recompute freely — the
  fusion-cheap region.

The ban is a **policy dial**, not a cost model. Set `ban=()` and even the
reduce is recomputed from the input — the boundary drops to zero saved bytes,
pure rematerialization, gradients still identical.

In [8]:
ck0 = checkpoint(crg.region, "loss", ban=(), names=crg.names)
print(f"ban=() (recompute all): saved {ck0.bytes_after} B, "
      f"recomputed {ck0.recomputed}")
g0 = run_named(ck0.region, cin, ck0.names)[crg.grads["x"]].to_numpy()
print("gradient still identical:", np.allclose(gj, g0, rtol=1e-12))

ban=() (recompute all): saved 0 B, recomputed ('sq', 'e', 'r')
gradient still identical: True


### GPT-2, with real numbers

Now the same cut on the GPT-2 joint. Two numbers come back, and they part
ways: the saved **boundary** collapses to 8% of naive — and the **peak** does
not move at all. Both are honest, and the gap between them is the lesson this
section ends on.

In [9]:
ck_g = checkpoint(rg.region, "zloss", names=rg.names)
pk_before = peak_memory(rg.region, m.inputs, names=rg.names).peak_bytes
pk_after = peak_memory(ck_g.region, m.inputs, names=ck_g.names).peak_bytes
print(f"saved boundary : {ck_g.bytes_before} B  ->  {ck_g.bytes_after} B   "
      f"({ck_g.bytes_after / ck_g.bytes_before:.0%})")
print(f"joint peak     : {pk_before} B  ->  {pk_after} B   "
      f"({pk_after / pk_before:.0%})")
print()
print("the cut lands on the contraction outputs — exactly what theory predicts:")
for v, b in ck_g.saved[:9]:
    print(f"    {v:<9}{b:>5} B")
print(f"    ...  {len(ck_g.saved)} saved tensors in all; "
      f"{len(ck_g.recomputed)} values recompute as clones")

saved boundary : 49472 B  ->  4064 B   (8%)
joint peak     : 18624 B  ->  18624 B   (100%)

the cut lands on the contraction outputs — exactly what theory predicts:
    mu          32 B
    reduce      32 B
    q          192 B
    k          192 B
    sc         256 B
    reduce1     64 B
    reduce2     64 B
    v          192 B
    cx         192 B
    ...  31 saved tensors in all; 133 values recompute as clones


Read the saved names: `mu` and the variance `reduce` (the LayerNorm
statistics), `q`/`k`/`v` (the QKV projections — matmul is
repeat·mul·**reduce**), `sc` (attention scores, a contraction), the two
softmax reduces (max and sum), then `cx` (the context), `o` (the output
projection), the MLP contractions, and the tied-head logits further down. The
cut saved the outputs of the banned reductions and recomputes all the cheap
pointwise/view glue between them — 133 values re-derive as clones.

And the peak? **It did not move.** At this toy scale the joint's high-water
mark sits in the late backward, where the resident weights and the
accumulating weight-gradients dominate — bytes no forward-activation cut can
touch. The caveat at the end of this notebook (min-cut minimizes the
*boundary*; no theorem connects it to the peak) is not a technicality: here
the two visibly decouple, and cashing the boundary win into a lower peak is
the schedule's business — pebbling, a later pass. Gradients are unchanged —
check the tied embedding and a weight deep in block 0.

In [10]:
ej = run_named(rg.region, m.inputs, rg.names)
ec = run_named(ck_g.region, m.inputs, ck_g.names)
for v in ("wte", "h.0.attn.wq"):
    ok = np.allclose(
        ec[rg.grads[v]].to_numpy(order=m.inputs[v].names),
        ej[rg.grads[v]].to_numpy(order=m.inputs[v].names),
        rtol=1e-10,
    )
    print(f"grad {v:<12} identical: {ok}")

grad wte          identical: True
grad h.0.attn.wq  identical: True


**The passes compose.** DCE prunes what you never asked for; checkpointing
plans what remains. Order them — prune first, then cut — and each contributes
the win it owns: DCE takes 480 B and 113 values off the joint; the cut then
collapses the surviving boundary to 4064 B. The composed *peak* equals the
DCE'd peak — at this scale the boundary win is real but the peak belongs to
the gradient accumulation, as measured above.

In [11]:
ck_dce = checkpoint(keep_wte, "zloss", names=rg.names)  # prune first, then cut
pk_dce = peak_memory(ck_dce.region, m.inputs, names=ck_dce.names).peak_bytes
print(f"DCE alone        : peak {peak_wte:>6} B   ({peak_wte / joint_peak:.0%} of joint)")
print(f"checkpoint alone : peak {pk_after:>6} B   ({pk_after / joint_peak:.0%} of joint)   "
      f"boundary {ck_g.bytes_before} -> {ck_g.bytes_after} B")
print(f"DCE + checkpoint : peak {pk_dce:>6} B   ({pk_dce / joint_peak:.0%} of joint)   "
      f"boundary {ck_dce.bytes_before} -> {ck_dce.bytes_after} B")

DCE alone        : peak  18144 B   (97% of joint)
checkpoint alone : peak  18624 B   (100% of joint)   boundary 49472 -> 4064 B
DCE + checkpoint : peak  18144 B   (97% of joint)   boundary 49472 -> 4064 B


## Move 3 — Segmented (checkpointed) fold adjoints

The min cut above works on a straight-line DAG. A `fold` (notebook 08) is a
*loop* whose body is itself a region, and its store-everything adjoint keeps
all T step-trajectories alive to feed the reverse pass — the loop analogue of
the saved-activations problem. `grad(..., fold_segments=K)` applies **Chen-style
uniform checkpointing** *inside* the fold adjoint: cut the time axis into K
equal segments, keep only the K segment-**boundary** states up front, and
recompute each segment's trajectory just-in-time during its own backward sweep.
So ~T/K + K states live instead of T.

Take the 1D FDTD leapfrog time-stepper from notebook 08 (two field states,
E and H; the step lifts from a plain function to a 15-value step region) and
run it for T=12 steps.

In [12]:
# 1D FDTD leapfrog: state = (E on 6 nodes, H on 5 edges); no per-step elements.
# The step is a plain function — C is closed over, n is structural — and it
# yields the two NEXT STATES in state order plus E1 again as the emitted
# value, so out=("emit",) makes the fold's result the E trajectory over t.
N, C = 6, 0.3


def fdtd_step(E, H, n: Literal[int]):
    dE = E.shift(x=-1).slice(x=(0, n - 1)) - E.slice(x=(0, n - 1))
    H1 = H + C * dE
    dH = H1.slice(x=(1, n - 1)) - H1.shift(x=1).slice(x=(1, n - 1))
    E1 = E + C * dH.pad(x=(0, n), fill=0.0)
    return E1, H1, E1


E = np.zeros(N); E[2] = 1.0          # a pulse
fin = {"E0": T(E, ("x",)), "H0": T(np.zeros(N - 1), ("x",))}
FDTD_STEP = lift_step(fdtd_step, E=fin["E0"].layout, H=fin["H0"].layout, n=N)


def fdtd_region(steps):
    # tl.fold over the lifted step, then the loss — sum of squared E over the
    # whole trajectory — authored with the Builder under the naming law
    b = Builder({**CORE_OPS, **TL_OPS})
    pE = b.param(0, tensor_type_of_layout(fin["E0"].layout))
    pH = b.param(1, tensor_type_of_layout(fin["H0"].layout))
    fold = b.emit("tl.fold", pE, pH, regions=(FDTD_STEP.region,), dim="t",
                  state=("E", "H"), element=(), out=("emit",), extent=(0, steps))
    E2 = b.emit("tl.pointwise", fold, fold, f="mul")
    loss = b.emit("tl.reduce", E2, f="sum", dims=("t", "x"))
    region = Region(params=(pE, pH), body=(b.emit("core.yield", loss),))
    names = region_names(region, ("E0", "H0"),
                         {id(fold): "Ef", id(E2): "E2", id(loss): "loss"})
    return region, names


prog, pnames = fdtd_region(12)
print(f"FDTD, T=12 steps; a {nvalues(FDTD_STEP.region)}-value step region; "
      "loss = sum of squared E over the whole trajectory")

FDTD, T=12 steps; a 15-value step region; loss = sum of squared E over the whole trajectory


Sweep K over the divisors of 12. `peak_memory` gives the bytes, `ops_count`
the recompute cost; the gradients must be identical to the store-everything
adjoint (K=None) at every K.

In [13]:
print(f"{'K':>4}  {'segments':>12}  {'peak B':>7}  {'ops (wtd)':>10}  {'dL/d(E0,H0)':>12}")
base = None
for K in (None, 2, 3, 4, 6, 12):
    kw = {} if K is None else {"fold_segments": K}
    rgf = grad(prog, "loss", fin, names=pnames, **kw)
    env = run_named(rgf.region, fin, rgf.names)
    res = {v: env[rgf.grads[v]].to_numpy() for v in ("E0", "H0")}
    peak = peak_memory(rgf.region, fin, names=rgf.names).peak_bytes
    ops = ops_count(rgf.region, names=rgf.names).weighted()
    if base is None:
        base, match = res, "baseline"
    else:
        ok = all(np.allclose(res[v], base[v], rtol=1e-9) for v in ("E0", "H0"))
        match = "identical" if ok else "DIFFERS"
    seg = "store all" if K is None else f"{K} x len {12 // K}"
    print(f"{str(K):>4}  {seg:>12}  {peak:>7}  {ops:>10.0f}  {match:>12}")

   K      segments   peak B   ops (wtd)   dL/d(E0,H0)
None     store all     2872        4043      baseline
   2     2 x len 6     2344        4415     identical


   3     3 x len 4     2256        4539     identical


   4     4 x len 3     2256        4601     identical
   6     6 x len 2     2344        4663     identical
  12    12 x len 1     2784        4725     identical


There is the curve. Peak falls **2872 → 2256 B** as segmentation kicks in,
then *rises* again at K=12 (2784 B) where the per-segment boundary-state
bookkeeping outweighs the trajectory it saves. The minimum sits at K=3–4, and
√12 ≈ 3.46 — **Chen's √T heuristic, reproduced from our own primitives**
rather than asserted. The `ops (wtd)` column is the price: recompute cost
climbs monotonically with K (4043 → 4725). And the gradients are identical at
every K — the segments change *when* work happens, never *what* it computes.

The segment count must divide the fold extent (K = a global uniform stride);
an indivisible K is refused loudly, not silently rounded.

In [14]:
try:
    grad(prog, "loss", fin, fold_segments=5, names=pnames)  # 12 % 5 != 0
except ValueError as e:
    print("refused:", e)

refused: fold_segments=5 must divide the fold extent 12 (pad the dim or pick a divisor)


**What remains**, kept honest. This is *uniform* (Chen-style)
checkpointing, not the optimal **binomial revolve** (Griewank & Walther,
O(log T) memory) — that is the next refinement. K is **global per `grad` call**,
not per-fold. K must **divide** T (pad the dim or pick a divisor first). And
one caveat that spans both Move 2 and Move 3: min-cut checkpointing minimizes
the *boundary* — whether the peak follows is the **schedule's** business, and
at this notebook's scale it did not move at all (GPT-2: 100%), because the
gradient accumulation owns the high-water mark. Direct-peak minimization
is schedule search / pebbling, a later pass. The ban set is a policy, not a
cost model (a tiny reduce is banned while a huge pointwise recomputes), and the
`.rc` clones break `name = value` until a value-numbering pass sees through
them.

---

## The L1 loop, and why it is the template

Every move here was the same three beats: **measure** with `peak_memory` /
`ops_count`, **transform** the region (DCE prune, min-cut rewrite, segmented
adjoint), **re-measure** to price the trade. Nothing was estimated — the layout
algebra is the exact alias theory, so the bytes are computed on the nose, and
the transforms are region-to-region rewrites whose gradients stay bit-for-bit
identical.

That loop is not special to L1. LEVELS.md lays the ladder above it: **L2**
storage (which bytes — liveness coloring, in-place), **L3** placement (where —
mesh-labeled dims, collectives from alignment diagnosis), **L4** kernels (who —
tiling, red-blue pebbling, the flash-attention derivation), **L5** schedule
(when — the timeline simulator). Each rung swaps in its own cost model and its
own transforms, but the shape is identical: *a white-box representation makes
the measurement exact, and an IR-to-IR rewrite pays down the number the
measurement exposed.* L1 is where that loop first closes; the rest of the ladder
just turns the same crank on a bigger machine.